# CS781 Phase 4 — Experiment 6 — v3 REVIEWED
## Knowledge Distillation: Model4 + LORMA Teacher → Model2 Student

**Teacher:** exact Phase-4 Exp2 Model4 + LORMA CJPE classifier.  
**Student:** exact Phase-1 Model2 decoder-only LM, fully task-fine-tuned by KD.

This reviewed version keeps the useful v2 design and adds failure guards that are especially
important for the final experiment:

- validates the reconstructed teacher on the 994-case clean DEV before trusting its logits;
- records and validates teacher-logit cache provenance (teacher/task/tokenizer/data hashes);
- rejects duplicate/incomplete teacher-logit caches;
- verifies the Phase-1 Model2 checkpoint/tokenizer pair with a next-token LM sanity check;
- adds a Model2 padding/causal-equivalence sanity check;
- logs student probability spread/positive rate during DEV evaluation;
- saves checkpoints atomically and records dataset hashes/configuration in the manifest;
- makes final refit non-finite/AMP-overflow handling consistent with main KD training.

Core KD methodology remains:
`35% supervised CE + 65% temperature-scaled KL`, `T=2.0`,
with full Model2 student fine-tuning and the same four hierarchical document views.


### Kaggle attachments required

Attach:

**Teacher**
- Phase-1 Model4 `eff_full_best*.pt`
- Exp2 `best_lorma_cjpe.pt`
- exact Model4 tokenizer JSON

**Student**
- Phase-1 Model2 `dec_full_best*.pt`
- exact Model2 tokenizer JSON

Optional:
- `teacher_logits_train_dev.npz` from a previous run of **this notebook**. If attached,
  v3 validates its IDs and provenance before reuse.

If auto-detection selects an ambiguous asset, set the corresponding override path in Cell 1.


In [1]:
# CELL 1 — imports/config
import os,re,gc,json,math,time,random,zipfile,subprocess,sys,hashlib,shutil,tempfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import f1_score,precision_score,recall_score
from huggingface_hub import hf_hub_download,login

SEED=1234
NOTEBOOK_VERSION="v3_REVIEWED"
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

def save_json(obj,path,**kw):
    path=Path(path)
    tmp=Path(str(path)+".tmp")
    with open(tmp,"w",encoding="utf-8") as f:
        json.dump(obj,f,**kw)
    os.replace(tmp,path)
def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)
if DEVICE.type=="cuda":
    print("GPU:",torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision("high")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN=None
assert HF_TOKEN,"HF_TOKEN missing in Kaggle Secrets"
login(token=HF_TOKEN)

HF_REPO="Exploration-Lab/CS-781-Capstone"
ROOT=Path("/kaggle/working/phase4_exp6_kd_m4lorma_to_m2")
ROOT.mkdir(parents=True,exist_ok=True)

# KD asset overrides (set explicit Kaggle paths only if auto-detection is ambiguous)
TEACHER_TASK_OVERRIDE=None
STUDENT_PHASE1_OVERRIDE=None
STUDENT_TOKENIZER_OVERRIDE=None
TEACHER_LOGITS_OVERRIDE=None

# Exact Model2 student
S_VOCAB_SIZE=24576
S_MAX_POSITIONS=2048
S_D_MODEL=640
S_N_LAYERS=13
S_N_HEADS=10
S_FFN_DIM=2560
S_DROPOUT=0.0

# KD optimization
KD_TEMPERATURE=2.0
KD_ALPHA=0.65
STUDENT_BACKBONE_LR=4.0e-5
STUDENT_HEAD_LR=3.0e-4
KD_NUM_EPOCHS=2
KD_EVAL_PER_EPOCH=3
KD_PATIENCE_EVALS=4
TEACHER_BATCH_SIZE=6
TEACHER_DEV_MIN_F1=0.60   # hard guard against a wrong teacher/cache; Exp2 was ~0.70 locally


# Exact Phase-1 Model4
VOCAB_SIZE=24576
MAX_POSITIONS=2560
D_MODEL=640
N_LAYERS=17
N_HEADS=10
N_KV_HEADS=2
FFN_DIM=2048
ROPE_BASE=10000.0
RMS_EPS=1e-5
DROPOUT=0.0
EXPECTED_TOKENIZER_SHA256="9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

# Long-document representation
N_CHUNKS=4
CHUNK_LEN=384
CHUNK_NAMES=["head","middle","evidence","tail"]

# LORMA — multiplicative low-rank adaptation
LORMA_R=8
LORMA_ALPHA=16
LORMA_DROPOUT=0.05
LORMA_LR=2.0e-4
LORMA_TARGETS=("q_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")

# Hierarchical task head
CHUNK_PROJ_DIM=384
HEAD_DROPOUT=0.15
HEAD_LR=5.0e-4

# Optimization
NUM_EPOCHS=2
EVAL_PER_EPOCH=3
PATIENCE_EVALS=4
MICRO_BATCH_SIZE=2
GRAD_ACCUM_STEPS=8
EVAL_BATCH_SIZE=6
WARMUP_RATIO=0.06
WEIGHT_DECAY=0.01
GRAD_CLIP=1.0
LABEL_SMOOTHING=0.02
USE_AMP=True
AMP_INIT_SCALE=4096.0
NUM_WORKERS=2

# DEV operating point
THRESHOLDS=np.round(np.arange(0.20,0.801,0.005),3)
THRESH_SMOOTH=9

# Same data normalization learned from audit
CASE_MODE="auto"
COLLAPSE_WS=True

# Final refit and one high-value submission
RUN_REFIT=True
REFIT_MATCH_ORIGINAL_SCHEDULE=True
ENSEMBLE_HELDOUT_REFIT=True

# Neutral phrases only choose an evidence region; they never map to label 0/1.
EVIDENCE_CUES=[
    "appeal is allowed","appeal allowed",
    "appeal is dismissed","appeal dismissed",
    "petition is allowed","petition allowed",
    "petition is dismissed","petition dismissed",
    "we allow","we dismiss",
    "is hereby allowed","is hereby dismissed",
    "set aside","acquitted","convicted",
    "in the result","accordingly",
]

print("chunks:",N_CHUNKS,"x",CHUNK_LEN)
print("effective document batch:",MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS)
print("LORMA:",LORMA_TARGETS,"r=",LORMA_R,"alpha=",LORMA_ALPHA,"dropout=",LORMA_DROPOUT)


Device: cuda
GPU: Tesla T4
chunks: 4 x 384
effective document batch: 16
LORMA: ('q_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj') r= 8 alpha= 16 dropout= 0.05


In [2]:
# CELL 2 — locate checkpoint/tokenizer
def all_files():
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root,_,files in os.walk(base):
            for fn in files:
                yield os.path.join(root,fn)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):
            h.update(b)
    return h.hexdigest()

files=list(all_files())
ckpts=[p for p in files if p.lower().endswith(".pt") and "eff_full_best" in os.path.basename(p).lower()]
toks=[p for p in files if os.path.basename(p).lower().startswith("tokenizer") and p.lower().endswith(".json")]

assert ckpts,"Attach Phase-1 Model4 eff_full_best*.pt"
assert toks,"Attach tokenizer.json"

PHASE1_CKPT=sorted(ckpts,key=lambda p:(sum(w in p.lower() for w in ["model4","phase1","efficient","eff_full_best"]),p),reverse=True)[0]
exact=[p for p in toks if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
assert exact,"Exact Phase-1 tokenizer not found"
TOKENIZER_FILE=exact[0]

print("checkpoint candidates:",len(ckpts))
for _p in ckpts: print("   -",_p)
if len(ckpts)>1: print("⚠️ several eval_full_best-style checkpoints found; verify the chosen one is Model4 (strict load below also checks).")
print("checkpoint:",PHASE1_CKPT)
print("tokenizer:",TOKENIZER_FILE)
print("SHA:",sha256_file(TOKENIZER_FILE))
# FIX: remember teacher assets; PHASE1_CKPT / TOKENIZER_FILE are overwritten later for the student
TEACHER_PHASE1=PHASE1_CKPT
TEACHER_TOKENIZER_FILE=TOKENIZER_FILE

# Teacher Phase4 task checkpoint
task_cands=[p for p in files if os.path.basename(p).lower()=="best_lorma_cjpe.pt"]
if TEACHER_TASK_OVERRIDE:
    TEACHER_TASK=str(TEACHER_TASK_OVERRIDE)
else:
    assert task_cands,"Attach best_lorma_cjpe.pt"
    TEACHER_TASK=sorted(task_cands,key=lambda p:(sum(w in p.lower() for w in ["model4","lorma","phase4"]),p),reverse=True)[0]

# Model2 Phase1 checkpoint
student_cands=[p for p in files if p.lower().endswith(".pt") and "dec_full_best" in os.path.basename(p).lower()]
if STUDENT_PHASE1_OVERRIDE:
    STUDENT_PHASE1=str(STUDENT_PHASE1_OVERRIDE)
else:
    assert student_cands,"Attach Model2 dec_full_best*.pt"
    STUDENT_PHASE1=sorted(student_cands,key=lambda p:(sum(w in p.lower() for w in ["model2","decoder","phase1"]),p),reverse=True)[0]

# Model2 tokenizer: prefer same directory as Model2 checkpoint.
if STUDENT_TOKENIZER_OVERRIDE:
    STUDENT_TOKENIZER=str(STUDENT_TOKENIZER_OVERRIDE)
else:
    sdir=str(Path(STUDENT_PHASE1).parent)
    def _ss(p):
        return (int(str(Path(p).parent)==sdir),
                len(os.path.commonpath([os.path.abspath(STUDENT_PHASE1),os.path.abspath(p)])),
                sum(w in p.lower() for w in ["model2","decoder","phase1"]),p)
    STUDENT_TOKENIZER=sorted(toks,key=_ss,reverse=True)[0]

cache_cands=[p for p in files if os.path.basename(p).lower()=="teacher_logits_train_dev.npz"]
if TEACHER_LOGITS_OVERRIDE:
    TEACHER_LOGITS_FILE=str(TEACHER_LOGITS_OVERRIDE)
elif cache_cands:
    TEACHER_LOGITS_FILE=sorted(
        cache_cands,
        key=lambda p:(sum(w in p.lower() for w in ["phase4","exp6","kd","m4lorma","model2"]),p),
        reverse=True
    )[0]
else:
    TEACHER_LOGITS_FILE=None
print("Teacher task candidates:",len(task_cands))
for _p in task_cands: print("   -",_p)
print("Student Phase1 candidates:",len(student_cands))
for _p in student_cands: print("   -",_p)
print("Teacher task:",TEACHER_TASK)
print("Teacher Phase1 SHA:",sha256_file(TEACHER_PHASE1))
print("Teacher task SHA  :",sha256_file(TEACHER_TASK))
print("Student Phase1:",STUDENT_PHASE1)
print("Student Phase1 SHA:",sha256_file(STUDENT_PHASE1))
print("Student tokenizer:",STUDENT_TOKENIZER)
print("Teacher-logit cache:",TEACHER_LOGITS_FILE)
print("Student tokenizer == teacher tokenizer:",sha256_file(STUDENT_TOKENIZER)==sha256_file(TEACHER_TOKENIZER_FILE),
      "(fine if Model2 really shares it; otherwise the LM-sanity check below will catch a wrong pick)")


checkpoint candidates: 1
   - /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
tokenizer: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
SHA: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d
Teacher task candidates: 1
   - /kaggle/input/datasets/aaryant23/phase2-model4/best_lorma_cjpe.pt
Student Phase1 candidates: 1
   - /kaggle/input/datasets/aaryant23/phase2-model4/dec_full_best (8).pt
Teacher task: /kaggle/input/datasets/aaryant23/phase2-model4/best_lorma_cjpe.pt
Teacher Phase1 SHA: 529cfef4605822c82075aee8254259ea8a42b91ec76ca41e32bd99ea7b61ed74
Teacher task SHA  : cbc3162985fb0bc7d393f6a1338e965d08e46c3942700acd70126a733d3fe3c9
Student Phase1: /kaggle/input/datasets/aaryant23/phase2-model4/dec_full_best (8).pt
Student Phase1 SHA: 7685624ff48e4c2fbf213f9be4f626173e52676b007b9bc50dc44b0323d3cc15
Student tokenizer: /kaggle/input/datasets/aaryant23/phase2-m

In [3]:
# CELL 3 — exact LegalBPE loader
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]
class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]
        self.itos=list(BPE_SPECIALS); self.stoi={t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word)
        while len(symbols)>1:
            best=None; rank=None
            for j in range(len(symbols)-1):
                rr=self._ranks.get((symbols[j],symbols[j+1]))
                if rr is not None and (rank is None or rr<rank):
                    best=(symbols[j],symbols[j+1]); rank=rr
            if best is None: break
            l,r=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==l and symbols[j+1]==r:
                    merged.append(l+r); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",str(text)):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok not in BPE_SPECIALS: buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        with open(path,encoding="utf-8") as f:
            obj=json.load(f)
        tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]
        tok.itos=list(obj["itos"]); tok.stoi={t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size=len(tok.itos); tok.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_FILE)
assert len(tokenizer)==VOCAB_SIZE
probe="IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(probe))==probe
print("Tokenizer PASS:",len(tokenizer))


Tokenizer PASS: 24576


In [4]:
# CELL 4 — download Phase4 + build clean splits (+ train/test text normalisation)
FNS={
 "train":"Phase_4_CJPE/Train/train.json",
 "dev":"Phase_4_CJPE/Train/dev.json",
 "test":"Phase_4_CJPE/Test/test.json",
}
paths={k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN) for k,v in FNS.items()}
DATA_SHA256={k:sha256_file(v) for k,v in paths.items()}
print("dataset SHA256:")
for _k,_h in DATA_SHA256.items(): print("  ",_k,_h)
train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])
RAW_ROW_COUNTS={"train":len(train_raw),"dev":len(dev_raw),"test":len(test_raw)}

def dedup_labeled(rows,name):
    seen={}; conflicts=set(); text_diff=0
    for r in rows:
        cid=str(r["id"]); y=int(r["label"]); txt=str(r["text"])
        assert y in (0,1)
        if cid in seen:
            if seen[cid]["label"]!=y: conflicts.add(cid)
            elif seen[cid]["text"]!=txt:
                text_diff+=1
                if len(txt)>len(seen[cid]["text"]): seen[cid]["text"]=txt   # keep the more complete version
        else:
            seen[cid]={"id":cid,"text":txt,"label":y}
    for c in conflicts: seen.pop(c,None)
    print(f"{name}: rows={len(rows)} unique={len(seen)+len(conflicts)} | same-ID different-text={text_diff} | label-conflict IDs dropped={len(conflicts)}")
    return list(seen.values())

train_records=dedup_labeled(train_raw,"train")
dev_unique=dedup_labeled(dev_raw,"dev")

# TEST: deliberately copy only id/text. Never read/use test label.
test_records=[{"id":str(r["id"]),"text":str(r["text"])} for r in test_raw]
test_ids={r["id"] for r in test_records}
assert len(test_ids)==len(test_records)

clean_dev=[r for r in dev_unique if r["id"] not in test_ids]
overlap_count=sum(r["id"] in test_ids for r in dev_unique)

if len(train_records)!=32305: print("⚠️ train unique count differs from audit:",len(train_records))
assert len(clean_dev)==994
assert len(test_records)==1517
if overlap_count==1517:
    print("✅ Older DEV layout: removed 1517 TEST-overlap IDs.")
elif overlap_count==0 and len(dev_unique)==994:
    print("✅ Current DEV layout already leakage-clean.")
else:
    print("⚠️ Unexpected DEV layout:",len(dev_unique),overlap_count)
assert not ({r["id"] for r in train_records}&{r["id"] for r in clean_dev})
assert not ({r["id"] for r in train_records}&test_ids)
assert not ({r["id"] for r in clean_dev}&test_ids)

# ---- text style diagnostics + normalisation (train looked mixed-case, dev/test lowercase) ----
def upper_ratio(t):
    t=t[:20000]; letters=sum(c.isalpha() for c in t); up=sum(c.isupper() for c in t)
    return up/max(letters,1)
_rng=random.Random(SEED)
def sample_ratio(rows,k=1500):
    idx=_rng.sample(range(len(rows)),min(k,len(rows)))
    return np.array([upper_ratio(rows[i]["text"]) for i in idx])
ur_tr,ur_dv,ur_te=sample_ratio(train_records),sample_ratio(clean_dev),sample_ratio(test_records)
print(f"uppercase-letter ratio  train median={np.median(ur_tr):.4f} (>2% in {100*np.mean(ur_tr>0.02):.1f}% of docs)"
      f" | clean-dev median={np.median(ur_dv):.4f} | test median={np.median(ur_te):.4f}")
print("newline-containing docs  train: {:.1f}% | dev: {:.1f}% | test: {:.1f}%".format(
    *[100*np.mean(["\n" in r["text"] for r in rows[:2000]]) for rows in (train_records,clean_dev,test_records)]))

LOWERCASE=(CASE_MODE=="lower") or (CASE_MODE=="auto" and float(np.median(ur_te))<0.01)
print("LOWERCASE:",LOWERCASE,"| COLLAPSE_WS:",COLLAPSE_WS)
def normalize_text(t):
    if LOWERCASE: t=t.lower()
    if COLLAPSE_WS: t=re.sub(r"\s+"," ",t).strip()
    return t
for rows in (train_records,clean_dev,test_records):
    for r in rows: r["text"]=normalize_text(r["text"])
del train_raw,dev_raw,test_raw,dev_unique; gc.collect()

print("TRAIN:",len(train_records),Counter(r["label"] for r in train_records))
print("CLEAN DEV:",len(clean_dev),Counter(r["label"] for r in clean_dev))
print("TEST:",len(test_records))
print("Excluded DEV↔TEST overlap:",overlap_count)
print("✅ TEST label/expert fields ignored.")


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

dataset SHA256:
   train a115cc722e9de8b68ac5d41179c12d86c2d537c0de8ff16412c44c6c8d1cce72
   dev aa2d80ae77f5d532fbc3f30e4dea2776700b8e92e71c603737d8f41a9d30b514
   test f3498fb832e90fe7264fcb6770918eb5261409e9a952d47a2979b6ed5b153b59
train: rows=37387 unique=32305 | same-ID different-text=0 | label-conflict IDs dropped=0
dev: rows=1988 unique=994 | same-ID different-text=0 | label-conflict IDs dropped=0
✅ Current DEV layout already leakage-clean.
uppercase-letter ratio  train median=0.0351 (>2% in 97.1% of docs) | clean-dev median=0.0000 | test median=0.0000
newline-containing docs  train: 0.0% | dev: 100.0% | test: 98.9%
LOWERCASE: True | COLLAPSE_WS: True
TRAIN: 32305 Counter({0: 18920, 1: 13385})
CLEAN DEV: 994 Counter({0: 497, 1: 497})
TEST: 1517
Excluded DEV↔TEST overlap: 0
✅ TEST label/expert fields ignored.


In [5]:
# CELL 5 — exact Model4 architecture
class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-5):
        super().__init__(); self.weight=nn.Parameter(torch.ones(dim)); self.eps=eps
    def forward(self,x):
        dtype=x.dtype; x32=x.float()
        x32=x32*torch.rsqrt(x32.pow(2).mean(-1,keepdim=True)+self.eps)
        return x32.to(dtype)*self.weight

class RotaryTable(nn.Module):
    def __init__(self,head_dim,max_positions,base=10000.0):
        super().__init__()
        inv=1.0/(base**(torch.arange(0,head_dim,2,dtype=torch.float32)/head_dim))
        freqs=torch.outer(torch.arange(max_positions,dtype=torch.float32),inv)
        self.register_buffer("cos_cached",freqs.cos(),persistent=False)
        self.register_buffer("sin_cached",freqs.sin(),persistent=False)
    def forward(self,position_ids):
        return self.cos_cached[position_ids].unsqueeze(1),self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x,cos,sin):
    B,H,T,D=x.shape; cos=cos.to(x.dtype); sin=sin.to(x.dtype)
    xp=x.view(B,H,T,D//2,2); x0,x1=xp[...,0],xp[...,1]
    return torch.stack((x0*cos-x1*sin,x0*sin+x1*cos),dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,n_kv_heads,dropout):
        super().__init__(); self.n_heads=n_heads; self.n_kv_heads=n_kv_heads
        self.head_dim=d_model//n_heads; self.kv_repeat=n_heads//n_kv_heads; self.dropout=dropout
        self.q_proj=nn.Linear(d_model,n_heads*self.head_dim,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.o_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x,cos,sin,key_valid=None):
        B,T,C=x.shape; H=self.n_heads; KV=self.n_kv_heads; D=self.head_dim
        q=self.q_proj(x).view(B,T,H,D).transpose(1,2)
        k=self.k_proj(x).view(B,T,KV,D).transpose(1,2)
        v=self.v_proj(x).view(B,T,KV,D).transpose(1,2)
        q=apply_rope(q,cos,sin); k=apply_rope(k,cos,sin)
        k=k.repeat_interleave(self.kv_repeat,dim=1); v=v.repeat_interleave(self.kv_repeat,dim=1)
        if key_valid is None:
            mask=None; causal=True
        else:
            qi=torch.arange(T,device=x.device)[:,None]
            kj=torch.arange(T,device=x.device)[None,:]
            mask=(kj<=qi)[None,None,:,:] & key_valid[:,None,None,:]
            mask=mask | (kj==qi)[None,None,:,:]
            causal=False
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=self.dropout if self.training else 0.0,is_causal=causal)
        return self.o_proj(y.transpose(1,2).contiguous().view(B,T,C))

class SwiGLU(nn.Module):
    def __init__(self,d_model,hidden_dim,dropout):
        super().__init__()
        self.gate_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.up_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.down_proj=nn.Linear(hidden_dim,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x))*self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1=RMSNorm(D_MODEL,RMS_EPS)
        self.attn=GQACausalSelfAttention(D_MODEL,N_HEADS,N_KV_HEADS,DROPOUT)
        self.norm2=RMSNorm(D_MODEL,RMS_EPS)
        self.ff=SwiGLU(D_MODEL,FFN_DIM,DROPOUT)
        self.resid_drop=nn.Dropout(DROPOUT)
    def forward(self,x,cos,sin,key_valid=None):
        x=x+self.resid_drop(self.attn(self.norm1(x),cos,sin,key_valid))
        x=x+self.ff(self.norm2(x))
        return x

class EfficientDecoderLM(nn.Module):
    def __init__(self):
        super().__init__(); self.vocab_size=VOCAB_SIZE; self.d_model=D_MODEL
        self.token_emb=nn.Embedding(VOCAB_SIZE,D_MODEL)
        self.drop=nn.Dropout(DROPOUT)
        self.rope=RotaryTable(D_MODEL//N_HEADS,MAX_POSITIONS,ROPE_BASE)
        self.blocks=nn.ModuleList([EfficientDecoderBlock() for _ in range(N_LAYERS)])
        self.final_norm=RMSNorm(D_MODEL,RMS_EPS)
    def forward_hidden(self,input_ids,key_valid=None):
        B,T=input_ids.shape; assert T<=MAX_POSITIONS
        pos=torch.arange(T,device=input_ids.device)[None,:].expand(B,T)
        cos,sin=self.rope(pos); x=self.drop(self.token_emb(input_ids))
        for block in self.blocks: x=block(x,cos,sin,key_valid)
        return self.final_norm(x)

def build_backbone():
    return EfficientDecoderLM()

bb=build_backbone()
obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
state=obj["model"] if isinstance(obj,dict) and "model" in obj else obj
bb.load_state_dict(state,strict=True)
print("Strict Model4 load PASS | params:",sum(p.numel() for p in bb.parameters())/1e6,"M")


Strict Model4 load PASS | params: 99.30944 M


In [6]:
# CELL 6 — inject LORMA + hierarchical classifier
class LoRMAPlusLinear(nn.Module):
    """
    Multiplicative low-rank adaptation of a frozen nn.Linear.

        z  = W0 x
        z' = z + (alpha/r) * B A dropout(z)

    Therefore W = (I + (alpha/r) BA) W0.

    B starts at zero, so the wrapped layer is exactly the frozen Phase-1
    projection at initialization.
    """
    def __init__(self,base,r=LORMA_R,alpha=LORMA_ALPHA,dropout=LORMA_DROPOUT):
        super().__init__()
        assert isinstance(base,nn.Linear)
        assert r>0
        self.base=base
        self.r=int(r)
        self.alpha=float(alpha)
        self.scale=self.alpha/self.r
        self.lorma_dropout=nn.Dropout(float(dropout))

        d_out=base.out_features
        self.lorma_A=nn.Parameter(torch.empty(self.r,d_out))
        self.lorma_B=nn.Parameter(torch.zeros(d_out,self.r))
        nn.init.kaiming_uniform_(self.lorma_A,a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad=False

    def forward(self,x):
        z=self.base(x)
        delta=F.linear(F.linear(self.lorma_dropout(z),self.lorma_A),self.lorma_B)
        return z+self.scale*delta

def inject_lorma(backbone):
    # Freeze every original Phase-1 parameter.
    for p in backbone.parameters():
        p.requires_grad=False

    injected=[]
    for bi,block in enumerate(backbone.blocks):
        # Attention projections
        for name in ("q_proj","v_proj","o_proj"):
            if name in LORMA_TARGETS:
                base=getattr(block.attn,name)
                setattr(block.attn,name,LoRMAPlusLinear(base))
                injected.append(f"blocks.{bi}.attn.{name}")

        # SwiGLU projections
        for name in ("gate_proj","up_proj","down_proj"):
            if name in LORMA_TARGETS:
                base=getattr(block.ff,name)
                setattr(block.ff,name,LoRMAPlusLinear(base))
                injected.append(f"blocks.{bi}.ff.{name}")

    lp=[(n,p) for n,p in backbone.named_parameters() if "lorma_" in n]
    assert len(lp)==2*len(injected),(len(lp),2*len(injected))
    assert all(p.requires_grad for _,p in lp)
    assert all(
        torch.count_nonzero(p.detach()).item()==0
        for n,p in lp if n.endswith("lorma_B")
    )

    expected=0
    for block in backbone.blocks:
        for modname in ("q_proj","v_proj","o_proj"):
            m=getattr(block.attn,modname)
            if isinstance(m,LoRMAPlusLinear):
                expected += 2*LORMA_R*m.base.out_features
        for modname in ("gate_proj","up_proj","down_proj"):
            m=getattr(block.ff,modname)
            if isinstance(m,LoRMAPlusLinear):
                expected += 2*LORMA_R*m.base.out_features
    actual=sum(p.numel() for _,p in lp)
    assert actual==expected,(actual,expected)

    print("Injected LORMA modules:",len(injected))
    print("LORMA trainable params:",f"{actual/1e6:.3f}M")
    return backbone

class HierarchicalCJPE(nn.Module):
    def __init__(self,backbone):
        super().__init__()
        self.backbone=backbone
        self.chunk_proj=nn.Sequential(
            nn.Linear(2*D_MODEL,CHUNK_PROJ_DIM),
            nn.GELU(),
            nn.LayerNorm(CHUNK_PROJ_DIM),
            nn.Dropout(HEAD_DROPOUT),
        )
        self.chunk_pos=nn.Embedding(N_CHUNKS,CHUNK_PROJ_DIM)
        self.attn_score=nn.Sequential(
            nn.Linear(CHUNK_PROJ_DIM,128),
            nn.Tanh(),
            nn.Linear(128,1),
        )
        self.fc=nn.Sequential(
            nn.Linear(2*CHUNK_PROJ_DIM,256),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(256,2),
        )

    def forward(self,input_ids,valid):
        B,K,T=input_ids.shape
        x=input_ids.view(B*K,T)
        m=valid.view(B*K,T)

        # Right padded + causal: valid positions cannot attend to future pads.
        h=self.backbone.forward_hidden(x,key_valid=None)

        mf=m.unsqueeze(-1).to(h.dtype)
        mean=(h*mf).sum(1)/mf.sum(1).clamp(min=1)
        last_idx=m.long().sum(1).clamp(min=1)-1
        last=h[torch.arange(B*K,device=h.device),last_idx]

        z=self.chunk_proj(torch.cat([last.float(),mean.float()],dim=-1)).view(B,K,-1)
        pos=torch.arange(K,device=z.device)[None,:]
        z=z+self.chunk_pos(pos)

        w=torch.softmax(self.attn_score(z).squeeze(-1),dim=-1)
        att=(z*w.unsqueeze(-1)).sum(1)
        mx=z.max(1).values
        return self.fc(torch.cat([att,mx],dim=-1)),w

bb=inject_lorma(bb)
model=HierarchicalCJPE(bb).to(DEVICE)

# Only LORMA + task head may train.
for n,p in model.backbone.named_parameters():
    if "lorma_" not in n:
        assert not p.requires_grad,n

lorma_params=sum(p.numel() for n,p in model.named_parameters() if "lorma_" in n and p.requires_grad)
head_params=sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and not n.startswith("backbone."))
total=sum(p.numel() for p in model.parameters())
print(f"LORMA trainable: {lorma_params/1e6:.3f}M")
print(f"Head trainable : {head_params/1e6:.3f}M")
print(f"Total trainable: {(lorma_params+head_params)/1e6:.3f}M / {total/1e6:.3f}M")


Injected LORMA modules: 102
LORMA trainable params: 1.671M
LORMA trainable: 1.671M
Head trainable : 0.741M
Total trainable: 2.412M / 101.722M


In [7]:
# CELL 7 — load validated Exp2 LORMA teacher task state
teacher_task_obj=torch.load(TEACHER_TASK,map_location="cpu",weights_only=False)
assert isinstance(teacher_task_obj,dict) and "trainable_state" in teacher_task_obj, \
    "Expected Exp2 best_lorma_cjpe.pt containing trainable_state."
teacher_state=teacher_task_obj["trainable_state"] if isinstance(teacher_task_obj,dict) and "trainable_state" in teacher_task_obj else teacher_task_obj
missing,unexpected=model.load_state_dict(teacher_state,strict=False)
assert not unexpected,unexpected
bad=[k for k in missing if ("lorma_" in k) or k.startswith(("chunk_proj.","chunk_pos.","attn_score.","fc."))]
assert not bad,bad
for p in model.parameters(): p.requires_grad=False
model.eval()
print("✅ Best Model4+LORMA teacher restored")
if isinstance(teacher_task_obj,dict):
    print("teacher DEV Macro-F1:",teacher_task_obj.get("best_macro_f1"))
    print("teacher threshold:",teacher_task_obj.get("best_threshold"))


✅ Best Model4+LORMA teacher restored
teacher DEV Macro-F1: 0.6994964429347799
teacher threshold: 0.43


In [8]:
# CELL 8 — tokenize/select HEAD-MIDDLE-EVIDENCE-TAIL (slice-based: no need to BPE-encode 400k-word cases)
SLICE_CHARS=CHUNK_LEN*8     # ~3k chars >> 384 BPE tokens

def _snap_forward(text,i):
    i=max(0,min(int(i),max(0,len(text)-1)))
    m=re.search(r"\s",text[i:i+400])
    return i+m.end() if m else i

def _enc(t):
    return tokenizer.encode(t)

def _window_start(text,center,width):
    s=max(0,int(center)-width//2)
    return _snap_forward(text,s)

def _last_evidence_position(text):
    low=text.lower()
    best=-1
    for cue in EVIDENCE_CUES:
        p=low.rfind(cue)
        if p>best:
            best=p
    return best

SELECT_STATS=Counter()   # diagnostics only

def select_chunks(text):
    """
    Four views of a long case:
      head / middle / evidence-or-late / tail.
    For short cases we derive all views from exact full BPE tokens.
    """
    n=len(text); W=SLICE_CHARS

    if n<=4*W:
        SELECT_STATS["short_doc"]+=1
        ids=_enc(text) or [tokenizer.unk_token_id]
        L=len(ids)

        if L<=CHUNK_LEN:
            ch=[ids,ids,ids,ids]
        else:
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(L-CHUNK_LEN,center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]

            # evidence view falls back to a late (75%) view for short docs.
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
        est=L

    else:
        # Prefix
        head=_enc(text[:W])

        # True character-centred middle slice.
        ms=_window_start(text,n*0.50,W)
        mid=_enc(text[ms:ms+W])

        # Evidence chunk: centre around the LAST neutral disposition cue.
        # If absent, use a late-document (75%) region.
        ep=_last_evidence_position(text)
        SELECT_STATS["long_doc"]+=1; SELECT_STATS["cue_found" if ep>=0 else "cue_missing"]+=1
        ec=(ep if ep>=0 else int(n*0.75))
        es=_window_start(text,ec,W)
        evidence=_enc(text[es:es+W])

        # Suffix
        ts=_snap_forward(text,max(0,n-W))
        tail=_enc(text[ts:])

        if min(len(head),len(mid),len(evidence),len(tail))<CHUNK_LEN:
            # Rare token-density/pathological fallback: exact full tokenisation.
            SELECT_STATS["fallback_full_tokenisation"]+=1
            ids=_enc(text) or [tokenizer.unk_token_id]
            L=len(ids)
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(max(0,L-CHUNK_LEN),center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
            est=L
        else:
            ch=[
                head[:CHUNK_LEN],
                mid[:CHUNK_LEN],
                evidence[:CHUNK_LEN],
                tail[-CHUNK_LEN:],
            ]
            est=int(n*len(head)/max(W,1))  # statistics only

    return [np.asarray(x,dtype=np.uint16) for x in ch],est

class CaseDataset(Dataset):
    def __init__(self,records,labeled):
        self.records=records; self.labeled=labeled; self.chunks=[]; self.full_lens=[]
        SELECT_STATS.clear(); t0=time.time()
        for i,r in enumerate(records):
            ch,L=select_chunks(r["text"]); self.chunks.append(ch); self.full_lens.append(L)
            if (i+1)%5000==0: print(f"tokenized {i+1}/{len(records)} elapsed={time.time()-t0:.0f}s",flush=True)
        print(f"done {len(records)} in {time.time()-t0:.0f}s | est. median BPE={np.median(self.full_lens):.0f} p95={np.percentile(self.full_lens,95):.0f}")
        print("   chunk-selection stats:",dict(SELECT_STATS))
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; d={"id":r["id"],"chunks":self.chunks[i]}
        if self.labeled: d["label"]=int(r["label"])
        return d

def collate_cases(batch):
    B=len(batch)
    ids=torch.full((B,N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((B,N_CHUNKS,CHUNK_LEN),dtype=torch.bool)
    for b,x in enumerate(batch):
        for k,c in enumerate(x["chunks"]):
            z=torch.from_numpy(c.astype(np.int64)); n=min(len(z),CHUNK_LEN)
            ids[b,k,:n]=z[:n]; valid[b,k,:n]=True
    out={"input_ids":ids,"valid":valid,"ids":[x["id"] for x in batch]}
    if "label" in batch[0]:
        out["labels"]=torch.tensor([x["label"] for x in batch],dtype=torch.long)
    return out


# Teacher TRAIN+DEV only (skipped entirely when cached teacher logits are available)
if TEACHER_LOGITS_FILE is None:
    t_train_ds=CaseDataset(train_records,True)
    t_dev_ds=CaseDataset(clean_dev,True)
    t_collate=collate_cases
    t_train_loader=DataLoader(t_train_ds,batch_size=TEACHER_BATCH_SIZE,shuffle=False,collate_fn=t_collate,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda")
    t_dev_loader=DataLoader(t_dev_ds,batch_size=TEACHER_BATCH_SIZE,shuffle=False,collate_fn=t_collate,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda")
else:
    print("Teacher-logit cache found -> skipping teacher-side tokenisation")


tokenized 5000/32305 elapsed=14s
tokenized 10000/32305 elapsed=24s
tokenized 15000/32305 elapsed=36s
tokenized 20000/32305 elapsed=50s
tokenized 25000/32305 elapsed=61s
tokenized 30000/32305 elapsed=72s
done 32305 in 76s | est. median BPE=4416 p95=20119
   chunk-selection stats: {'long_doc': 15940, 'cue_found': 12469, 'cue_missing': 3471, 'short_doc': 16365}
done 994 in 3s | est. median BPE=6232 p95=21137
   chunk-selection stats: {'long_doc': 657, 'cue_found': 480, 'cue_missing': 177, 'short_doc': 337}


In [9]:
# CELL 9 — cache teacher logits, validate strong teacher, then free teacher
LOGITS_OUT=ROOT/"teacher_logits_train_dev.npz"

TEACHER_CACHE_META={
    "format_version":"phase4_exp6_teacher_logits_v3",
    "teacher_phase1_sha256":sha256_file(TEACHER_PHASE1),
    "teacher_task_sha256":sha256_file(TEACHER_TASK),
    "teacher_tokenizer_sha256":sha256_file(TEACHER_TOKENIZER_FILE),
    "train_sha256":DATA_SHA256["train"],
    "dev_sha256":DATA_SHA256["dev"],
    "lowercase":bool(LOWERCASE),
    "collapse_ws":bool(COLLAPSE_WS),
    "n_chunks":int(N_CHUNKS),
    "chunk_len":int(CHUNK_LEN),
}

def load_teacher_cache(path):
    z=np.load(path,allow_pickle=False)
    assert "id" in z.files and "logits" in z.files,"Teacher cache must contain id/logits."
    ids=z["id"].astype(str)
    lg=z["logits"].astype(np.float32)
    assert lg.shape==(len(ids),2) and np.isfinite(lg).all(),"Bad teacher-cache logits"
    assert len(ids)==len(set(ids.tolist())),"Duplicate IDs in teacher-logit cache"

    # v3 caches carry provenance. Legacy caches can still be checked by the strong DEV-F1 guard below.
    if "meta_json" in z.files:
        meta=json.loads(str(z["meta_json"].item()))
        for k,v in TEACHER_CACHE_META.items():
            assert meta.get(k)==v,(
                f"Teacher cache provenance mismatch for {k}: "
                f"cache={meta.get(k)!r}, current={v!r}. "
                "Remove/override the stale cache so logits are regenerated."
            )
        print("✅ teacher-cache provenance PASS")
    else:
        print("⚠️ Legacy teacher cache has no provenance metadata; DEV-F1 validation will decide whether it is safe.")

    return {str(i):lg[j] for j,i in enumerate(ids)}

if TEACHER_LOGITS_FILE:
    teacher_map=load_teacher_cache(TEACHER_LOGITS_FILE)
    print("✅ Reused teacher logits:",len(teacher_map),TEACHER_LOGITS_FILE)
else:
    @torch.no_grad()
    def collect_teacher(loader,name):
        ids=[];lgs=[];t0=time.time()
        for j,b in enumerate(loader,1):
            x=b["input_ids"].to(DEVICE,non_blocking=True)
            v=b["valid"].to(DEVICE,non_blocking=True)
            with torch.amp.autocast(
                "cuda",dtype=torch.float16,
                enabled=(DEVICE.type=="cuda" and USE_AMP)
            ):
                lg,_=model(x,v)
            assert bool(torch.isfinite(lg).all()),"Non-finite teacher logits"
            ids.extend(b["ids"])
            lgs.append(lg.float().cpu().numpy())
            if j%500==0 or j==len(loader):
                print(
                    f"teacher {name}: {j}/{len(loader)} "
                    f"elapsed={(time.time()-t0)/60:.1f}m",
                    flush=True
                )
        return ids,np.concatenate(lgs)

    a,la=collect_teacher(t_train_loader,"TRAIN")
    b,lb=collect_teacher(t_dev_loader,"DEV")
    ids=np.asarray(a+b,dtype=str)
    logits=np.concatenate([la,lb]).astype(np.float32)

    assert len(ids)==33299,(len(ids),33299)
    assert len(ids)==len(set(ids.tolist()))
    np.savez_compressed(
        LOGITS_OUT,
        id=ids,
        logits=logits,
        meta_json=np.asarray(json.dumps(TEACHER_CACHE_META,sort_keys=True)),
    )
    teacher_map={str(i):logits[j] for j,i in enumerate(ids)}
    print("✅ teacher logits saved:",LOGITS_OUT)

needed={r["id"] for r in train_records+clean_dev}
cache_ids=set(teacher_map)
assert needed==cache_ids,(
    f"Teacher-cache ID set mismatch: "
    f"missing={len(needed-cache_ids)} extra={len(cache_ids-needed)}"
)

# ---- teacher sanity: refuse to distill from a wrong/weak reconstructed teacher ----
def _p1_np(lg):
    lg=lg.astype(np.float64)
    lg-=lg.max(-1,keepdims=True)
    e=np.exp(lg)
    return e[:,1]/e.sum(-1)

_dy=np.array([r["label"] for r in clean_dev],dtype=np.int64)
_dp=_p1_np(np.stack([teacher_map[r["id"]] for r in clean_dev]))
_fs=np.array([
    f1_score(
        _dy,(_dp>=t).astype(np.int64),
        average="macro",labels=[0,1],zero_division=0
    )
    for t in THRESHOLDS
])
_teacher_best_i=int(np.argmax(_fs))
_teacher_dev_f1=float(_fs[_teacher_best_i])
_teacher_thr=float(THRESHOLDS[_teacher_best_i])

print(
    f"teacher clean-DEV Macro-F1={_teacher_dev_f1:.6f} "
    f"@ threshold={_teacher_thr:.3f}"
)
assert _teacher_dev_f1>=TEACHER_DEV_MIN_F1,(
    f"Teacher validation FAILED: DEV Macro-F1={_teacher_dev_f1:.4f} "
    f"< {TEACHER_DEV_MIN_F1:.2f}. "
    "Do not distill: wrong teacher checkpoint/tokenizer/cache is likely selected."
)

_ty=np.array([r["label"] for r in train_records],dtype=np.int64)
_tp=_p1_np(np.stack([teacher_map[r["id"]] for r in train_records]))
print(
    f"teacher TRAIN acc@0.5={((_tp>=.5)==_ty).mean():.4f} | "
    f"mean max-prob TRAIN={np.maximum(_tp,1-_tp).mean():.3f} "
    f"DEV={np.maximum(_dp,1-_dp).mean():.3f}"
)
print("✅ strong teacher validation PASS")

# Teacher GPU objects no longer needed.
for _n in [
    "model","bb","t_train_loader","t_dev_loader","t_train_ds","t_dev_ds",
    "teacher_task_obj","teacher_state","state","obj"
]:
    globals().pop(_n,None)
gc.collect()
if DEVICE.type=="cuda":
    torch.cuda.empty_cache()
print("✅ teacher freed; student stage begins")


teacher TRAIN: 500/5385 elapsed=1.8m
teacher TRAIN: 1000/5385 elapsed=3.5m
teacher TRAIN: 1500/5385 elapsed=5.3m
teacher TRAIN: 2000/5385 elapsed=7.1m
teacher TRAIN: 2500/5385 elapsed=8.9m
teacher TRAIN: 3000/5385 elapsed=10.7m
teacher TRAIN: 3500/5385 elapsed=12.4m
teacher TRAIN: 4000/5385 elapsed=14.2m
teacher TRAIN: 4500/5385 elapsed=16.0m
teacher TRAIN: 5000/5385 elapsed=17.8m
teacher TRAIN: 5385/5385 elapsed=19.1m
teacher DEV: 166/166 elapsed=0.6m
✅ teacher logits saved: /kaggle/working/phase4_exp6_kd_m4lorma_to_m2/teacher_logits_train_dev.npz
teacher clean-DEV Macro-F1=0.702714 @ threshold=0.415
teacher TRAIN acc@0.5=0.7450 | mean max-prob TRAIN=0.726 DEV=0.705
✅ strong teacher validation PASS
✅ teacher freed; student stage begins


In [10]:
# CELL 10 — switch to exact Model2 tokenizer
TOKENIZER_FILE=STUDENT_TOKENIZER
tokenizer=LegalBPE.load_json(TOKENIZER_FILE)
assert len(tokenizer)==S_VOCAB_SIZE
VOCAB_SIZE=S_VOCAB_SIZE
MAX_POSITIONS=S_MAX_POSITIONS
D_MODEL=S_D_MODEL
N_LAYERS=S_N_LAYERS
N_HEADS=S_N_HEADS
FFN_DIM=S_FFN_DIM
DROPOUT=S_DROPOUT
PHASE1_CKPT=STUDENT_PHASE1
probe="IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(probe))==probe
print("✅ Model2 tokenizer loaded:",TOKENIZER_FILE)


✅ Model2 tokenizer loaded: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json


In [12]:
# CELL 11 — exact Phase-1 Model2 architecture + strict checkpoint load
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_positions):
        super().__init__()
        pos=torch.arange(max_positions,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_positions,d_model,dtype=torch.float32)
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe,persistent=False)
    def forward(self,position_ids): return self.pe[position_ids]

class CausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__()
        assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,past_kv=None,use_cache=False,key_valid=None):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2)
        q=q.transpose(1,2); k=k.transpose(1,2); v=v.transpose(1,2)
        past_len=0
        if past_kv is not None:
            pk,pv=past_kv; past_len=pk.size(2)
            k=torch.cat([pk,k],dim=2); v=torch.cat([pv,v],dim=2)
        attn_mask=None; is_causal=False
        if key_valid is not None:
            key_mask=key_valid[:,None,None,:]
            if past_len==0 and T>1:
                causal=torch.ones((T,T),dtype=torch.bool,device=x.device).tril()
                attn_mask=key_mask & causal[None,None,:,:]
                eye=torch.eye(T,dtype=torch.bool,device=x.device)[None,None,:,:]
                attn_mask=attn_mask|eye
            else: attn_mask=key_mask
        elif past_kv is None:
            is_causal=True
        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        y=self.out_proj(y)
        return y,((k,v) if use_cache else None)

class DecoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.attn=CausalSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,ffn_dim),nn.GELU(),
            nn.Linear(ffn_dim,d_model),nn.Dropout(dropout),
        )
        self.resid_drop=nn.Dropout(dropout)
    def forward(self,x,past_kv=None,use_cache=False,key_valid=None):
        a,new_kv=self.attn(self.ln1(x),past_kv=past_kv,use_cache=use_cache,key_valid=key_valid)
        x=x+self.resid_drop(a)
        x=x+self.ff(self.ln2(x))
        return x,new_kv

class DecoderOnlyLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,ffn_dim,dropout,max_positions):
        super().__init__()
        self.vocab_size=vocab_size; self.d_model=d_model; self.max_positions=max_positions
        self.token_emb=nn.Embedding(vocab_size,d_model)
        self.pos=SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop=nn.Dropout(dropout)
        self.blocks=nn.ModuleList([DecoderBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_layers)])
        self.final_ln=nn.LayerNorm(d_model)
        self.apply(self._init_weights)
        scaled_std=0.02/math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight,mean=0.0,std=scaled_std)
            nn.init.normal_(block.ff[2].weight,mean=0.0,std=scaled_std)
    def _init_weights(self,m):
        if isinstance(m,nn.Linear):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
    def forward_hidden(self,input_ids,past_key_values=None,use_cache=False,position_ids=None,key_valid=None):
        B,T=input_ids.shape
        past_len=0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len+T>self.max_positions:
            raise ValueError(f"context {past_len+T} exceeds max_positions={self.max_positions}")
        if position_ids is None:
            position_ids=torch.arange(past_len,past_len+T,device=input_ids.device)[None,:].expand(B,T)
        x=self.token_emb(input_ids)+self.pos(position_ids).to(self.token_emb.weight.dtype)
        x=self.drop(x)
        new_cache=[] if use_cache else None
        for i,block in enumerate(self.blocks):
            past=None if past_key_values is None else past_key_values[i]
            x,kv=block(x,past_kv=past,use_cache=use_cache,key_valid=key_valid)
            if use_cache:new_cache.append(kv)
        x=self.final_ln(x)
        return x,new_cache
    def project(self,hidden): return F.linear(hidden,self.token_emb.weight)

def build_backbone():
    return DecoderOnlyLM(
        vocab_size=VOCAB_SIZE,d_model=D_MODEL,n_layers=N_LAYERS,n_heads=N_HEADS,
        ffn_dim=FFN_DIM,dropout=DROPOUT,max_positions=MAX_POSITIONS,
    )

obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
state=obj["model"] if isinstance(obj,dict) and "model" in obj else obj
assert tuple(state["token_emb.weight"].shape)==(VOCAB_SIZE,D_MODEL)
assert len(tokenizer)==VOCAB_SIZE
bb=build_backbone()
bb.load_state_dict(state,strict=True)
n_params=sum(p.numel() for p in bb.parameters())
assert 75_000_000<=n_params<=85_000_000,n_params
print(f"Strict Model2 load PASS | params={n_params/1e6:.3f}M")
if isinstance(obj,dict) and "config" in obj: print("checkpoint config:",obj["config"])
del obj,state; gc.collect()
# ---- FIXED Model2 checkpoint/tokenizer next-token sanity ----
@torch.no_grad()
def model2_phase1_lm_sanity(n_docs=10):
    global bb

    # FIX: backbone was still on CPU while inputs were on CUDA
    bb = bb.to(DEVICE)
    bb.eval()

    losses = []

    for r in train_records[:max(n_docs*3, 30)]:
        ids = tokenizer.encode(r["text"][:12000])

        if len(ids) < 96:
            continue

        ids = ids[:min(256, len(ids))]

        x = torch.tensor(
            [ids[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [ids[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda" and USE_AMP)
        ):
            h, _ = bb.forward_hidden(
                x,
                key_valid=None
            )

            lg = bb.project(h).float()

        loss = F.cross_entropy(
            lg.reshape(-1, lg.size(-1)),
            y.reshape(-1)
        )

        losses.append(float(loss))

        if len(losses) >= n_docs:
            break

    assert losses, "Model2 LM sanity had no usable documents."

    m = float(np.mean(losses))

    print(
        f"Phase-1 Model2 LM sanity loss={m:.3f} | "
        f"ppl={math.exp(min(m,20)):.1f} | "
        f"chance ln(V)={math.log(VOCAB_SIZE):.2f}"
    )

    if m > 7.0:
        print(
            "⚠️ Model2 LM loss is unusually high; "
            "verify STUDENT_TOKENIZER_OVERRIDE."
        )

    assert m < 8.5, (
        f"Model2 checkpoint/tokenizer sanity FAILED: loss={m:.3f}. "
        "Likely wrong Model2 tokenizer/checkpoint pairing."
    )

    print("✅ Model2 checkpoint/tokenizer sanity PASS")


model2_phase1_lm_sanity()

Strict Model2 load PASS | params=79.736M
checkpoint config: {'architecture': 'decoder_only_causal', 'vocab_size': 24576, 'd_model': 640, 'n_layers': 13, 'n_heads': 10, 'ffn_dim': 2560, 'dropout': 0.0, 'max_positions': 2048, 'seq_len': 1536, 'prompt_p10_tokens': 1050, 'prompt_p95_tokens': 1139, 'expected_cont_tokens': 384, 'prompt_split_min': 1050, 'prompt_split_max': 1139}
Phase-1 Model2 LM sanity loss=3.134 | ppl=23.0 | chance ln(V)=10.11
✅ Model2 checkpoint/tokenizer sanity PASS


In [13]:
# CELL 12 — full Model2 KD student + hierarchical classifier
class HierarchicalCJPE(nn.Module):
    def __init__(self,backbone):
        super().__init__();self.backbone=backbone
        self.chunk_proj=nn.Sequential(
            nn.Linear(2*D_MODEL,CHUNK_PROJ_DIM),nn.GELU(),
            nn.LayerNorm(CHUNK_PROJ_DIM),nn.Dropout(HEAD_DROPOUT)
        )
        self.chunk_pos=nn.Embedding(N_CHUNKS,CHUNK_PROJ_DIM)
        self.attn_score=nn.Sequential(nn.Linear(CHUNK_PROJ_DIM,128),nn.Tanh(),nn.Linear(128,1))
        self.fc=nn.Sequential(nn.Linear(2*CHUNK_PROJ_DIM,256),nn.GELU(),nn.Dropout(HEAD_DROPOUT),nn.Linear(256,2))
    def forward(self,input_ids,valid):
        B,K,T=input_ids.shape;x=input_ids.view(B*K,T);m=valid.view(B*K,T)
        h=self.backbone.forward_hidden(x,key_valid=None)[0]
        mf=m.unsqueeze(-1).to(h.dtype);mean=(h*mf).sum(1)/mf.sum(1).clamp(min=1)
        li=m.long().sum(1).clamp(min=1)-1;last=h[torch.arange(B*K,device=h.device),li]
        z=self.chunk_proj(torch.cat([last.float(),mean.float()],-1)).view(B,K,-1)
        z=z+self.chunk_pos(torch.arange(K,device=z.device)[None,:])
        w=torch.softmax(self.attn_score(z).squeeze(-1),-1)
        return self.fc(torch.cat([(z*w.unsqueeze(-1)).sum(1),z.max(1).values],-1)),w

model=HierarchicalCJPE(bb).to(DEVICE)
print("KD student total/trainable params:",
      sum(p.numel() for p in model.parameters())/1e6,
      sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6)


@torch.no_grad()
def verify_student_padding_equivalence():
    """
    Because chunks are right-padded and the student is causal, valid positions should be
    unaffected by future pad tokens when key_valid=None.
    """
    model.eval()
    b=next(iter(dev_loader)) if "dev_loader" in globals() else None
    if b is None:
        print("padding-equivalence sanity deferred until loaders exist")
        return
    x=b["input_ids"][:1,:1].reshape(1,-1).to(DEVICE)
    v=b["valid"][:1,:1].reshape(1,-1).to(DEVICE)
    n=int(v.sum())
    assert n>8
    with torch.amp.autocast(
        "cuda",dtype=torch.float16,
        enabled=(DEVICE.type=="cuda" and USE_AMP)
    ):
        h_full,_=model.backbone.forward_hidden(x,key_valid=None)
        h_trim,_=model.backbone.forward_hidden(x[:,:n],key_valid=None)
    d=float((h_full[:,:n]-h_trim).abs().max())
    print("Model2 right-padding causal equivalence max |Δ|:",d)
    assert d<0.10,d
    model.train()

# Called after student loaders are constructed in Cell 13.


KD student total/trainable params: 80.476675 80.476675


In [14]:
# CELL 13 — tokenize/select HEAD-MIDDLE-EVIDENCE-TAIL (slice-based: no need to BPE-encode 400k-word cases)
SLICE_CHARS=CHUNK_LEN*8     # ~3k chars >> 384 BPE tokens

def _snap_forward(text,i):
    i=max(0,min(int(i),max(0,len(text)-1)))
    m=re.search(r"\s",text[i:i+400])
    return i+m.end() if m else i

def _enc(t):
    return tokenizer.encode(t)

def _window_start(text,center,width):
    s=max(0,int(center)-width//2)
    return _snap_forward(text,s)

def _last_evidence_position(text):
    low=text.lower()
    best=-1
    for cue in EVIDENCE_CUES:
        p=low.rfind(cue)
        if p>best:
            best=p
    return best

SELECT_STATS=Counter()   # diagnostics only

def select_chunks(text):
    """
    Four views of a long case:
      head / middle / evidence-or-late / tail.
    For short cases we derive all views from exact full BPE tokens.
    """
    n=len(text); W=SLICE_CHARS

    if n<=4*W:
        SELECT_STATS["short_doc"]+=1
        ids=_enc(text) or [tokenizer.unk_token_id]
        L=len(ids)

        if L<=CHUNK_LEN:
            ch=[ids,ids,ids,ids]
        else:
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(L-CHUNK_LEN,center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]

            # evidence view falls back to a late (75%) view for short docs.
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
        est=L

    else:
        # Prefix
        head=_enc(text[:W])

        # True character-centred middle slice.
        ms=_window_start(text,n*0.50,W)
        mid=_enc(text[ms:ms+W])

        # Evidence chunk: centre around the LAST neutral disposition cue.
        # If absent, use a late-document (75%) region.
        ep=_last_evidence_position(text)
        SELECT_STATS["long_doc"]+=1; SELECT_STATS["cue_found" if ep>=0 else "cue_missing"]+=1
        ec=(ep if ep>=0 else int(n*0.75))
        es=_window_start(text,ec,W)
        evidence=_enc(text[es:es+W])

        # Suffix
        ts=_snap_forward(text,max(0,n-W))
        tail=_enc(text[ts:])

        if min(len(head),len(mid),len(evidence),len(tail))<CHUNK_LEN:
            # Rare token-density/pathological fallback: exact full tokenisation.
            SELECT_STATS["fallback_full_tokenisation"]+=1
            ids=_enc(text) or [tokenizer.unk_token_id]
            L=len(ids)
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(max(0,L-CHUNK_LEN),center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
            est=L
        else:
            ch=[
                head[:CHUNK_LEN],
                mid[:CHUNK_LEN],
                evidence[:CHUNK_LEN],
                tail[-CHUNK_LEN:],
            ]
            est=int(n*len(head)/max(W,1))  # statistics only

    return [np.asarray(x,dtype=np.uint16) for x in ch],est

class CaseDataset(Dataset):
    def __init__(self,records,labeled):
        self.records=records; self.labeled=labeled; self.chunks=[]; self.full_lens=[]
        SELECT_STATS.clear(); t0=time.time()
        for i,r in enumerate(records):
            ch,L=select_chunks(r["text"]); self.chunks.append(ch); self.full_lens.append(L)
            if (i+1)%5000==0: print(f"tokenized {i+1}/{len(records)} elapsed={time.time()-t0:.0f}s",flush=True)
        print(f"done {len(records)} in {time.time()-t0:.0f}s | est. median BPE={np.median(self.full_lens):.0f} p95={np.percentile(self.full_lens,95):.0f}")
        print("   chunk-selection stats:",dict(SELECT_STATS))
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; d={"id":r["id"],"chunks":self.chunks[i]}
        if self.labeled: d["label"]=int(r["label"])
        return d

def collate_cases(batch):
    B=len(batch)
    ids=torch.full((B,N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((B,N_CHUNKS,CHUNK_LEN),dtype=torch.bool)
    for b,x in enumerate(batch):
        for k,c in enumerate(x["chunks"]):
            z=torch.from_numpy(c.astype(np.int64)); n=min(len(z),CHUNK_LEN)
            ids[b,k,:n]=z[:n]; valid[b,k,:n]=True
    out={"input_ids":ids,"valid":valid,"ids":[x["id"] for x in batch]}
    if "label" in batch[0]:
        out["labels"]=torch.tensor([x["label"] for x in batch],dtype=torch.long)
    return out

train_ds=CaseDataset(train_records,True)
dev_ds=CaseDataset(clean_dev,True)
test_ds=CaseDataset(test_records,False)

g=torch.Generator(); g.manual_seed(SEED)
_kw=dict(collate_fn=collate_cases,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0)
train_loader=DataLoader(train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=g,**_kw)
dev_loader=DataLoader(dev_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)

print("batches:",len(train_loader),len(dev_loader),len(test_loader))

# Verify the v2 fast causal-kernel optimisation on a real padded batch.
# Valid tokens in a causal model cannot attend to future right-padding.
@torch.no_grad()
def verify_fast_causal_equivalence():
    b=next(iter(dev_loader))
    x=b["input_ids"][:1].to(DEVICE)
    v=b["valid"][:1].to(DEVICE)
    B,K,T=x.shape
    xf=x.view(B*K,T); vf=v.view(B*K,T)
    model.backbone.eval()
    with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
        h_mask,_=model.backbone.forward_hidden(xf,key_valid=vf)
        h_fast,_=model.backbone.forward_hidden(xf,key_valid=None)
    d=(h_mask[vf]-h_fast[vf]).abs().max().item()
    print("fast causal-kernel valid-token max |Δ|:",d)
    assert d < 5e-2,(
        "Fast causal path deviates unexpectedly; restore key_valid masking."
    )
    model.train()

verify_fast_causal_equivalence()

# Sanity: the frozen Phase-1 LM should predict next tokens far better than chance (ln V = 10.1).
# The student is full fine-tuned (no adapters), and nothing has been trained yet -> this is exactly the Phase-1 model.
@torch.no_grad()
def phase1_lm_sanity(n_docs=16):
    model.backbone.eval(); losses=[]
    for i in range(min(n_docs,len(train_ds))):
        ids=torch.from_numpy(train_ds.chunks[i][1].astype(np.int64))[None].to(DEVICE)   # middle view
        if ids.size(1)<32: continue
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
            h,_=model.backbone.forward_hidden(ids)
            lg=model.backbone.project(h[:,:-1]).float()
        losses.append(F.cross_entropy(lg[0],ids[0,1:]).item())
    model.train()
    if not losses: print("LM sanity skipped (docs too short)"); return
    m=float(np.mean(losses))
    print(f"Phase-1 LM sanity: next-token loss={m:.3f} (ppl {math.exp(m):.1f}) | chance={math.log(VOCAB_SIZE):.2f}")
    if m>7.0:
        print("⚠️ LM loss is unusually high; tokenizer/text normalization may not match Phase-1 Model2.")
    assert m<8.5, (
        f"Phase-1 LM sanity FAILED: mean next-token loss={m:.3f}. "
        "Most likely the wrong tokenizer.json was selected. "
        "Set STUDENT_TOKENIZER_OVERRIDE to the Model2 tokenizer."
    )
phase1_lm_sanity()

verify_student_padding_equivalence()


tokenized 5000/32305 elapsed=15s
tokenized 10000/32305 elapsed=26s
tokenized 15000/32305 elapsed=39s
tokenized 20000/32305 elapsed=53s
tokenized 25000/32305 elapsed=65s
tokenized 30000/32305 elapsed=77s
done 32305 in 81s | est. median BPE=4416 p95=20119
   chunk-selection stats: {'long_doc': 15940, 'cue_found': 12469, 'cue_missing': 3471, 'short_doc': 16365}
done 994 in 3s | est. median BPE=6232 p95=21137
   chunk-selection stats: {'long_doc': 657, 'cue_found': 480, 'cue_missing': 177, 'short_doc': 337}
done 1517 in 4s | est. median BPE=6505 p95=22861
   chunk-selection stats: {'long_doc': 1053, 'cue_found': 772, 'short_doc': 464, 'cue_missing': 281}
batches: 16153 166 253
fast causal-kernel valid-token max |Δ|: 0.0
Phase-1 LM sanity: next-token loss=2.656 (ppl 14.2) | chance=10.11
Model2 right-padding causal equivalence max |Δ|: 0.0


In [15]:
# CELL 14 — KD optimizer + metrics
counts=Counter(r["label"] for r in train_records)
class_w=torch.tensor([len(train_records)/(2*counts[i]) for i in [0,1]],dtype=torch.float32,device=DEVICE)
class_w/=class_w.mean()
print("class weights:",class_w.tolist())

def _no_decay(n,p):
    # standard practice: no weight decay on biases, LayerNorm/RMSNorm gains, embeddings
    return p.ndim<=1 or "token_emb" in n or "chunk_pos" in n
def make_groups(m):
    out=[]
    for gname,lr in (("backbone",STUDENT_BACKBONE_LR),("head",STUDENT_HEAD_LR)):
        ps=[(n,p) for n,p in m.named_parameters() if p.requires_grad and (n.startswith("backbone.")==(gname=="backbone"))]
        out.append({"params":[p for n,p in ps if not _no_decay(n,p)],"lr":lr,"weight_decay":WEIGHT_DECAY,"name":gname})
        out.append({"params":[p for n,p in ps if _no_decay(n,p)],"lr":lr,"weight_decay":0.0,"name":gname+"_nodecay"})
    return out
groups=make_groups(model)
try:optimizer=torch.optim.AdamW(groups,fused=(DEVICE.type=="cuda"),betas=(.9,.95))
except Exception:optimizer=torch.optim.AdamW(groups,betas=(.9,.95))

NUM_EPOCHS=KD_NUM_EPOCHS;EVAL_PER_EPOCH=KD_EVAL_PER_EPOCH;PATIENCE_EVALS=KD_PATIENCE_EVALS
updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS);total_updates=NUM_EPOCHS*updates_per_epoch
warmup=max(1,int(total_updates*WARMUP_RATIO))
def lr_mult(step):
    if step<warmup:return max(1e-8,step/warmup)
    p=(step-warmup)/max(1,total_updates-warmup);return .5*(1+math.cos(math.pi*min(max(p,0),1)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_mult)
amp_enabled=USE_AMP and DEVICE.type=="cuda"
scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE,growth_interval=2000)

def kd_loss(sl,tl,y,weights):
    hard=F.cross_entropy(sl.float(),y,weight=weights,label_smoothing=LABEL_SMOOTHING)
    T=KD_TEMPERATURE
    soft=F.kl_div(F.log_softmax(sl.float()/T,-1),F.softmax(tl.float()/T,-1),reduction="batchmean")*(T*T)
    return (1-KD_ALPHA)*hard+KD_ALPHA*soft,hard,soft
def macro_f1(y,p):return f1_score(y,p,average="macro",labels=[0,1],zero_division=0)
def search_threshold(y,prob,smooth=THRESH_SMOOTH):
    fs=np.array([macro_f1(y,(prob>=t).astype(np.int64)) for t in THRESHOLDS])
    k=max(1,int(smooth))
    if k>1:
        if k%2==0:k+=1
        pad=k//2;sm=np.convolve(np.pad(fs,(pad,pad),mode="edge"),np.ones(k)/k,mode="valid")
    else:sm=fs
    i=int(np.argmax(sm));t=float(THRESHOLDS[i]);return float(fs[i]),t,(prob>=t).astype(np.int64),float(fs.max())
@torch.no_grad()
def predict_logits(m,loader):
    was=m.training;m.eval();lgs=[];ys=[];ids=[];ats=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE);v=b["valid"].to(DEVICE)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):lg,a=m(x,v)
        lgs.append(lg.float().cpu().numpy());ats.append(a.float().cpu().numpy());ids+=b["ids"]
        if "labels" in b:ys.append(b["labels"].numpy())
    m.train(was);return np.concatenate(lgs),(np.concatenate(ys) if ys else None),ids,np.concatenate(ats)
def _p1(lg):
    lg=lg.astype(np.float64);lg-=lg.max(-1,keepdims=True);e=np.exp(lg);return (e[:,1]/e.sum(-1)).astype(np.float32)
def _logit(p):
    p=np.clip(np.asarray(p,dtype=np.float64),1e-6,1-1e-6);return np.log(p/(1-p))
def predict_probs(m,loader):
    lg,y,ids,a=predict_logits(m,loader);return _p1(lg),y,ids,a
print("updates/epoch",updates_per_epoch,"total",total_updates)


class weights: [0.8286643028259277, 1.1713356971740723]
updates/epoch 2020 total 4040


In [16]:
# CELL 15 — KD train / clean DEV selection
BEST=ROOT/"best_kd_model2_cjpe.pt";TMP=Path(str(BEST)+".tmp")
if TMP.exists(): TMP.unlink()
history=[];best_f1=-1.;best_thr=.5;best_frac=None;bad=0;global_update=0;overflows=0
optimizer.zero_grad(set_to_none=True)
eval_steps=[]
for j in range(1,EVAL_PER_EPOCH+1):
    s=int(round(len(train_loader)*j/EVAL_PER_EPOCH))
    if s<len(train_loader):s=max(GRAD_ACCUM_STEPS,(s//GRAD_ACCUM_STEPS)*GRAD_ACCUM_STEPS)
    else:s=len(train_loader)
    eval_steps.append(s)
eval_steps=sorted(set(eval_steps));print("DEV eval batches",eval_steps)
stop=False
for epoch in range(1,NUM_EPOCHS+1):
    model.train();t0=time.time();rt=rh=rs=0.;nr=0;good=0
    for step,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE);v=b["valid"].to(DEVICE);y=b["labels"].to(DEVICE)
        tl=torch.tensor(np.stack([teacher_map[str(i)] for i in b["ids"]]),dtype=torch.float32,device=DEVICE)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            sl,_=model(x,v);tot,hard,soft=kd_loss(sl,tl,y,class_w);loss=tot/GRAD_ACCUM_STEPS
        if math.isfinite(float(tot.detach())):
            scaler.scale(loss).backward();good+=1;nr+=1;rt+=float(tot);rh+=float(hard);rs+=float(soft)
        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            if good:
                scaler.unscale_(optimizer);gn=torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                old=float(scaler.get_scale())
                # FIX: always step()+update(). scaler.step() itself skips non-finite grads; the old
                # "skip update() if grad-norm is inf" branch left the optimizer in UNSCALED state and
                # crashed the next unscale_() with a RuntimeError.
                scaler.step(optimizer);scaler.update();new=float(scaler.get_scale());optimizer.zero_grad(set_to_none=True)
                if new>=old:scheduler.step();global_update+=1
                else:overflows+=1
            else:optimizer.zero_grad(set_to_none=True)
            good=0
        if step%500==0 or step==len(train_loader):
            el=time.time()-t0;eta=(el/max(step,1))*(len(train_loader)-step)
            print(f"epoch={epoch} batch={step}/{len(train_loader)} KD={rt/max(nr,1):.4f} hard={rh/max(nr,1):.4f} soft={rs/max(nr,1):.4f} elapsed={el/60:.1f}m ETA={eta/60:.1f}m",flush=True)
        if step in eval_steps:
            prob,ydev,idsdev,att=predict_probs(model,dev_loader);f,t,pred,grid=search_threshold(ydev,prob)
            row={"epoch":epoch,"epoch_frac":(epoch-1)+step/len(train_loader),"batch":step,"macro_f1":f,"threshold":t,
                 "grid_best_macro_f1":grid,"f1_at_0.5":float(macro_f1(ydev,(prob>=.5).astype(np.int64))),
                 "kd_loss":rt/max(nr,1),"hard_loss":rh/max(nr,1),"soft_loss":rs/max(nr,1),
                 "prob_mean":float(prob.mean()),"prob_std":float(prob.std()),
                 "prob_min":float(prob.min()),"prob_max":float(prob.max()),
                 "positive_rate":float((prob>=t).mean()),
                 "positive_rate_05":float((prob>=.5).mean()),
                 "attention_mean":att.mean(0).tolist()}
            history.append(row);save_json(history,ROOT/"history.json",indent=2);print("DEV",row)
            if f>best_f1+1e-12:
                best_f1=f;best_thr=t;best_frac=row["epoch_frac"];bad=0
                torch.save({"model_state":model.state_dict(),"best_macro_f1":best_f1,"best_threshold":best_thr,"best_epoch_frac":best_frac,
                            "kd_temperature":KD_TEMPERATURE,"kd_alpha":KD_ALPHA,
                            "student_backbone_lr":STUDENT_BACKBONE_LR,"student_head_lr":STUDENT_HEAD_LR,
                            "teacher_dev_macro_f1":_teacher_dev_f1,
                            "teacher_dev_threshold":_teacher_thr,
                            "seed":SEED},TMP);os.replace(TMP,BEST)
                print("✅ NEW BEST",best_f1,best_thr)
            else:bad+=1
            if bad>=PATIENCE_EVALS:stop=True;break
            model.train()
    if stop:break
assert BEST.exists();print("DONE",best_f1,best_thr,best_frac,"| skipped (overflow) updates:",overflows)


DEV eval batches [5384, 10768, 16153]


/tmp/ipykernel_49/3093025564.py:22: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  scaler.scale(loss).backward();good+=1;nr+=1;rt+=float(tot);rh+=float(hard);rs+=float(soft)


epoch=1 batch=500/16153 KD=0.4271 hard=0.6979 soft=0.2812 elapsed=0.9m ETA=27.2m
epoch=1 batch=1000/16153 KD=0.4239 hard=0.7047 soft=0.2728 elapsed=1.7m ETA=26.2m
epoch=1 batch=1500/16153 KD=0.4217 hard=0.7036 soft=0.2699 elapsed=2.6m ETA=25.3m
epoch=1 batch=2000/16153 KD=0.4080 hard=0.6935 soft=0.2543 elapsed=3.5m ETA=24.4m
epoch=1 batch=2500/16153 KD=0.3947 hard=0.6866 soft=0.2375 elapsed=4.3m ETA=23.5m
epoch=1 batch=3000/16153 KD=0.3833 hard=0.6797 soft=0.2238 elapsed=5.2m ETA=22.7m
epoch=1 batch=3500/16153 KD=0.3726 hard=0.6714 soft=0.2117 elapsed=6.0m ETA=21.8m
epoch=1 batch=4000/16153 KD=0.3637 hard=0.6663 soft=0.2008 elapsed=6.9m ETA=21.0m
epoch=1 batch=4500/16153 KD=0.3563 hard=0.6616 soft=0.1918 elapsed=7.8m ETA=20.1m
epoch=1 batch=5000/16153 KD=0.3502 hard=0.6585 soft=0.1841 elapsed=8.6m ETA=19.2m
DEV {'epoch': 1, 'epoch_frac': 0.333312697331765, 'batch': 5384, 'macro_f1': 0.6275841677197758, 'threshold': 0.39, 'grid_best_macro_f1': 0.6353365628230334, 'f1_at_0.5': 0.58414762

In [17]:
# CELL 16 — exact supplied Phase-4 evaluator on clean DEV
best_obj=torch.load(BEST,map_location="cpu",weights_only=False)
model.load_state_dict(best_obj["model_state"],strict=True)
best_thr=float(best_obj["best_threshold"])

dev_prob,dev_y,dev_ids,dev_att=predict_probs(model,dev_loader)
dev_pred=(dev_prob>=best_thr).astype(np.int64)

truth=[{"id":str(i),"gold":int(y)} for i,y in zip(dev_ids,dev_y)]
pred=[{"id":str(i),"pred":int(p)} for i,p in zip(dev_ids,dev_pred)]

EVALUATOR_CODE=r'''#!/usr/bin/env python3
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score
def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out
def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True)
    a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e:
        print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''

with tempfile.TemporaryDirectory() as td:
    tp=Path(td)/"truth.json"
    pp=Path(td)/"prediction.json"
    ep=Path(td)/"evaluate_phase4.py"
    save_json(truth,tp)
    save_json(pred,pp)
    ep.write_text(EVALUATOR_CODE,encoding="utf-8")
    res=subprocess.run(
        [sys.executable,str(ep),"--truth",str(tp),"--prediction",str(pp)],
        capture_output=True,text=True,
    )
    print(res.stdout)
    if res.stderr: print(res.stderr)
    assert res.returncode==0

print("✅ Exact Phase-4 evaluator PASS")
np.savez_compressed(
    ROOT/"dev_probs.npz",
    prob=dev_prob.astype(np.float32),
    y=dev_y.astype(np.int64),
    id=np.asarray(dev_ids,dtype=str),   # loads without allow_pickle
    threshold=np.asarray([best_thr],dtype=np.float32),
    chunk_attention=dev_att.astype(np.float32),
)


Examples: 994
Macro-F1: 0.707568
Precision: 0.730512
Recall: 0.659960

✅ Exact Phase-4 evaluator PASS


In [18]:
# CELL 17 — final KD refit on TRAIN+clean DEV
def fresh_model():
    b=build_backbone();o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False);st=o["model"] if isinstance(o,dict) and "model" in o else o;b.load_state_dict(st,strict=True)
    return HierarchicalCJPE(b).to(DEVICE)
refit_model=None;REFIT_SEED=SEED+1
if RUN_REFIT:
    optimizer.state.clear(); gc.collect(); torch.cuda.empty_cache() if DEVICE.type=="cuda" else None   # free Adam states of the first student
    all_records=train_records+clean_dev;all_ds=torch.utils.data.ConcatDataset([train_ds,dev_ds])
    seed_everything(REFIT_SEED);gg=torch.Generator();gg.manual_seed(REFIT_SEED)
    all_loader=DataLoader(all_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=gg,collate_fn=collate_cases,
                          num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0)
    refit_model=fresh_model();rgs=make_groups(refit_model)
    try:ropt=torch.optim.AdamW(rgs,fused=(DEVICE.type=="cuda"),betas=(.9,.95))
    except Exception:ropt=torch.optim.AdamW(rgs,betas=(.9,.95))
    target_batches=max(GRAD_ACCUM_STEPS,int(round(float(best_obj["best_epoch_frac"])*len(all_loader)/GRAD_ACCUM_STEPS))*GRAD_ACCUM_STEPS)
    upe=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS);horizon=NUM_EPOCHS*upe;rw=max(1,int(horizon*WARMUP_RATIO))
    def rmult(step):
        if step<rw:return max(1e-8,step/rw)
        p=(step-rw)/max(1,horizon-rw);return .5*(1+math.cos(math.pi*min(max(p,0),1)))
    rsched=torch.optim.lr_scheduler.LambdaLR(ropt,rmult);rscaler=torch.amp.GradScaler("cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE)
    ac=Counter(r["label"] for r in all_records);cw=torch.tensor([len(all_records)/(2*ac[i]) for i in [0,1]],dtype=torch.float32,device=DEVICE);cw/=cw.mean()
    ropt.zero_grad(set_to_none=True);seen=0;good=0;t0=time.time();refit_overflows=0;refit_nonfinite=0
    print("refit target",target_batches)
    while seen<target_batches:
        for b in all_loader:
            if seen>=target_batches:break
            x=b["input_ids"].to(DEVICE);v=b["valid"].to(DEVICE);y=b["labels"].to(DEVICE)
            tl=torch.tensor(np.stack([teacher_map[str(i)] for i in b["ids"]]),dtype=torch.float32,device=DEVICE)
            with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
                sl,_=refit_model(x,v);tot,_,_=kd_loss(sl,tl,y,cw);loss=tot/GRAD_ACCUM_STEPS
            seen+=1
            if math.isfinite(float(tot.detach())):
                rscaler.scale(loss).backward();good+=1
            else:
                refit_nonfinite+=1

            if seen%GRAD_ACCUM_STEPS==0 or seen>=target_batches:
                if good:
                    rscaler.unscale_(ropt)
                    torch.nn.utils.clip_grad_norm_(refit_model.parameters(),GRAD_CLIP)
                    _o=float(rscaler.get_scale())
                    rscaler.step(ropt)
                    rscaler.update()
                    _n=float(rscaler.get_scale())
                    ropt.zero_grad(set_to_none=True)
                    if _n>=_o:
                        rsched.step()
                    else:
                        refit_overflows+=1
                else:
                    ropt.zero_grad(set_to_none=True)
                good=0
            if seen%500==0 or seen>=target_batches:
                rate=(time.time()-t0)/max(seen,1);print(f"refit {seen}/{target_batches} ETA={(target_batches-seen)*rate/60:.1f}m",flush=True)
    FINAL=ROOT/"final_refit_kd_model2_cjpe.pt"
    FTMP=Path(str(FINAL)+".tmp")
    if FTMP.exists(): FTMP.unlink()
    torch.save({
        "model_state":refit_model.state_dict(),
        "source_best_epoch_frac":best_obj["best_epoch_frac"],
        "kd_temperature":KD_TEMPERATURE,
        "kd_alpha":KD_ALPHA,
        "teacher_dev_macro_f1":_teacher_dev_f1,
        "refit_seed":REFIT_SEED,
        "refit_overflows":refit_overflows,
        "refit_nonfinite":refit_nonfinite,
    },FTMP)
    os.replace(FTMP,FINAL)
    print("✅ refit done | overflows=",refit_overflows,"nonfinite=",refit_nonfinite)


refit target 33296
refit 500/33296 ETA=56.6m
refit 1000/33296 ETA=55.7m
refit 1500/33296 ETA=54.8m
refit 2000/33296 ETA=54.0m
refit 2500/33296 ETA=53.0m
refit 3000/33296 ETA=52.2m
refit 3500/33296 ETA=51.3m
refit 4000/33296 ETA=50.4m
refit 4500/33296 ETA=49.6m
refit 5000/33296 ETA=48.7m
refit 5500/33296 ETA=47.8m
refit 6000/33296 ETA=47.0m
refit 6500/33296 ETA=46.1m
refit 7000/33296 ETA=45.2m
refit 7500/33296 ETA=44.4m
refit 8000/33296 ETA=43.5m
refit 8500/33296 ETA=42.6m
refit 9000/33296 ETA=41.7m
refit 9500/33296 ETA=40.9m
refit 10000/33296 ETA=40.0m
refit 10500/33296 ETA=39.2m
refit 11000/33296 ETA=38.3m
refit 11500/33296 ETA=37.4m
refit 12000/33296 ETA=36.6m
refit 12500/33296 ETA=35.7m
refit 13000/33296 ETA=34.9m
refit 13500/33296 ETA=34.0m
refit 14000/33296 ETA=33.1m
refit 14500/33296 ETA=32.3m
refit 15000/33296 ETA=31.4m
refit 15500/33296 ETA=30.6m
refit 16000/33296 ETA=29.7m
refit 16500/33296 ETA=28.8m
refit 17000/33296 ETA=28.0m
refit 17500/33296 ETA=27.1m
refit 18000/33296 ETA

In [19]:
# CELL 18 — TEST inference + label-free calibration
held_lg,_,held_ids,held_att=predict_logits(model,test_loader)
assert held_ids==[r["id"] for r in test_records] and len(held_ids)==1517
held_prob=_p1(held_lg)

refit_prob=None; refit_thr=None; test_refit_agreement=None
if refit_model is not None:
    refit_lg,_,rid,_=predict_logits(refit_model,test_loader)
    assert rid==held_ids
    refit_prob=_p1(refit_lg)
    held_rate=float((held_prob>=best_thr).mean())
    refit_thr=float(np.clip(np.quantile(refit_prob,1-held_rate),.05,.95))
    test_refit_agreement=float(((held_prob>=best_thr)==(refit_prob>=refit_thr)).mean())
    print(f"heldout positive rate={held_rate:.3f} | refit thr={refit_thr:.3f} | agreement={test_refit_agreement:.4f}")

np.savez_compressed(
    ROOT/"test_probs.npz",heldout=held_prob.astype(np.float32),
    refit=(refit_prob.astype(np.float32) if refit_prob is not None else np.array([],dtype=np.float32)),
    id=np.asarray(held_ids,dtype=str),
    heldout_threshold=np.asarray([best_thr],dtype=np.float32),
    refit_threshold=np.asarray([refit_thr if refit_thr is not None else np.nan],dtype=np.float32),
)


heldout positive rate=0.477 | refit thr=0.435 | agreement=0.8734


In [20]:
# CELL 19 — three high-value Codabench ZIPs
official_ids=[r["id"] for r in test_records]
assert len(official_ids)==len(set(official_ids))==1517

def write_zip(filename,labels):
    labels=np.asarray(labels,dtype=np.int64)
    rec=[{"id":str(cid),"pred":int(y)} for cid,y in zip(official_ids,labels)]
    assert len(rec)==1517 and len({r["id"] for r in rec})==1517
    assert all(type(r["pred"]) is int and r["pred"] in (0,1) for r in rec)
    zp=ROOT/filename
    with tempfile.TemporaryDirectory() as td:
        jp=Path(td)/"prediction.json"
        save_json(rec,jp)
        with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z:
            z.write(jp,arcname="prediction.json")
    with zipfile.ZipFile(zp,"r") as z:
        assert z.namelist()==["prediction.json"],z.namelist()
        inside=json.loads(z.read("prediction.json").decode("utf-8"))
        assert len(inside)==1517
        assert {r["id"] for r in inside}==set(official_ids)
    print(f"✅ {filename} | positive={labels.mean():.4f} | {Counter(labels.tolist())}")
    return labels

if refit_prob is not None and ENSEMBLE_HELDOUT_REFIT:
    margin=.5*(_logit(held_prob)-_logit(best_thr))+.5*(_logit(refit_prob)-_logit(refit_thr))
    y_best=write_zip("submission_best.zip",(margin>=0).astype(np.int64))
else:
    y_best=write_zip("submission_best.zip",(held_prob>=best_thr).astype(np.int64))

y_held=write_zip("submission_heldout.zip",(held_prob>=best_thr).astype(np.int64))

if refit_prob is not None:
    margin_h70_r30=.70*(_logit(held_prob)-_logit(best_thr)) + .30*(_logit(refit_prob)-_logit(refit_thr))
    y_hedge=write_zip("submission_margin_h70_r30.zip",(margin_h70_r30>=0).astype(np.int64))
else:
    y_hedge=None

disagree_best_held=float(np.mean(y_best!=y_held))
disagree_best_hedge=(None if y_hedge is None else float(np.mean(y_best!=y_hedge)))
disagree_held_hedge=(None if y_hedge is None else float(np.mean(y_held!=y_hedge)))
print("best vs heldout disagreement:",disagree_best_held)
if y_hedge is not None:
    print("best vs 70/30 hedge disagreement:",disagree_best_hedge)
    print("heldout vs 70/30 hedge disagreement:",disagree_held_hedge)


✅ submission_best.zip | positive=0.4845 | Counter({0: 782, 1: 735})
✅ submission_heldout.zip | positive=0.4766 | Counter({0: 794, 1: 723})
✅ submission_margin_h70_r30.zip | positive=0.4832 | Counter({0: 784, 1: 733})
best vs heldout disagreement: 0.06591957811470006
best vs 70/30 hedge disagreement: 0.03164139749505603
heldout vs 70/30 hedge disagreement: 0.034278180619644036


In [21]:
# CELL 20 — manifest
manifest={
 "phase":4,"experiment":6,"method":"Knowledge Distillation","notebook_version":NOTEBOOK_VERSION,
 "teacher":"Model4+LORMA Exp2","student":"Phase1 Model2 full fine-tuning",
 "kd_temperature":KD_TEMPERATURE,"kd_alpha":KD_ALPHA,
 "student_backbone_lr":STUDENT_BACKBONE_LR,"student_head_lr":STUDENT_HEAD_LR,
 "teacher_validated_dev_macro_f1":_teacher_dev_f1,"teacher_validated_dev_threshold":_teacher_thr,
 "teacher_task_sha256":sha256_file(TEACHER_TASK),"teacher_phase1_sha256":sha256_file(TEACHER_PHASE1),"teacher_tokenizer_sha256":sha256_file(TEACHER_TOKENIZER_FILE),
 "student_phase1_sha256":sha256_file(PHASE1_CKPT),"student_tokenizer_sha256":sha256_file(TOKENIZER_FILE),
 "dataset_sha256":DATA_SHA256,"raw_dataset_rows":RAW_ROW_COUNTS,
 "train_unique":len(train_records),"clean_dev":len(clean_dev),"test":len(test_records),"excluded_dev_test_overlap":overlap_count,
 "best_dev_macro_f1":float(best_obj["best_macro_f1"]),"best_threshold":float(best_obj["best_threshold"]),"best_epoch_frac":float(best_obj["best_epoch_frac"]),
 "refit_run":bool(refit_model is not None),"refit_threshold":None if refit_thr is None else float(refit_thr),
 "test_refit_agreement":test_refit_agreement,"test_positive_rate_best":float(np.mean(y_best)),
 "test_labels_used":False,"overlap_dev_labels_used":False,"seed":SEED,"torch_version":torch.__version__
}
save_json(manifest,ROOT/"manifest.json",indent=2);print(json.dumps(manifest,indent=2))
print("\nESSENTIAL OUTPUTS")
for fn in ["teacher_logits_train_dev.npz","best_kd_model2_cjpe.pt","final_refit_kd_model2_cjpe.pt","dev_probs.npz","test_probs.npz","history.json","manifest.json","submission_best.zip","submission_heldout.zip","submission_margin_h70_r30.zip"]:
    print(("✅" if (ROOT/fn).exists() else "—"),fn)


{
  "phase": 4,
  "experiment": 6,
  "method": "Knowledge Distillation",
  "notebook_version": "v3_REVIEWED",
  "teacher": "Model4+LORMA Exp2",
  "student": "Phase1 Model2 full fine-tuning",
  "kd_temperature": 2.0,
  "kd_alpha": 0.65,
  "student_backbone_lr": 4e-05,
  "student_head_lr": 0.0003,
  "teacher_validated_dev_macro_f1": 0.7027135260631314,
  "teacher_validated_dev_threshold": 0.415,
  "teacher_task_sha256": "cbc3162985fb0bc7d393f6a1338e965d08e46c3942700acd70126a733d3fe3c9",
  "teacher_phase1_sha256": "529cfef4605822c82075aee8254259ea8a42b91ec76ca41e32bd99ea7b61ed74",
  "teacher_tokenizer_sha256": "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d",
  "student_phase1_sha256": "7685624ff48e4c2fbf213f9be4f626173e52676b007b9bc50dc44b0323d3cc15",
  "student_tokenizer_sha256": "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d",
  "dataset_sha256": {
    "train": "a115cc722e9de8b68ac5d41179c12d86c2d537c0de8ff16412c44c6c8d1cce72",
    "dev": "aa2d80ae7

## Submission order
1. `submission_best.zip`
2. `submission_margin_h70_r30.zip`
3. `submission_heldout.zip`

Save `teacher_logits_train_dev.npz`; it lets you restart later without rerunning the teacher.
